## Environment Setup

In [1]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes datasets qwen-vl-utils sentence-transformers json-repair nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 69.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 78.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 81.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 80.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 69.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 65.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 82.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/28

## Imports & Global Configurations

In [1]:
import os
# Force Hugging Face to download massive weights to the persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

In [2]:
import os
import re
import json
import torch
import torch.nn as nn
import numpy as np
import nltk
from PIL import Image
from transformers import (
    AutoProcessor, AutoModelForImageTextToText, 
    AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
)
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from sentence_transformers import SentenceTransformer
import json_repair

nltk.download("punkt", quiet=True)

# --- Global Configurations ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAFE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

QWEN_BASE_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
QWEN_LORA_PATH = "/workspace/qwen_lora_final/qwen_lora_final"
DEBERTA_MODEL_ID = "cross-encoder/nli-deberta-v3-large"
DEBERTA_CKPT_PATH = "best_nli_verifier.pt"
FLAN_T5_ID = "google/flan-t5-base"

## Load All Models into VRAM

In [3]:
print("1. Loading Stage 1: Qwen-VL + LoRA...")
qwen_processor = AutoProcessor.from_pretrained(QWEN_BASE_ID)
base_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_BASE_ID, device_map="auto", torch_dtype=SAFE_DTYPE, attn_implementation="sdpa"
)
qwen_model = PeftModel.from_pretrained(base_model, QWEN_LORA_PATH).merge_and_unload()
qwen_model.eval()

print("2. Loading Stage 2: MiniLM Semantic Retriever...")
semantic_retriever = SentenceTransformer("all-MiniLM-L6-v2")

print("3. Loading Stage 2: DeBERTa NLI Verifier...")
class NLIVerifierWithAttention(nn.Module):
    def __init__(self, model_name=DEBERTA_MODEL_ID, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden // 2, 1)
        )

    def forward(self, input_ids, attention_mask, return_attentions=True):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_attentions=return_attentions)
        cls_rep = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_rep).squeeze(-1)
        if return_attentions:
            stacked = torch.stack(outputs.attentions, dim=0)
            return logits, stacked.mean(dim=0).mean(dim=1)
        return logits

nli_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_ID)
nli_model = NLIVerifierWithAttention().to(DEVICE, dtype=SAFE_DTYPE)
# nli_model.load_state_dict(torch.load(DEBERTA_CKPT_PATH, map_location="cpu")) # Uncomment when weights are ready
nli_model.eval()

print("4. Loading Stage 2: Flan-T5 Rationale Generator...")
t5_tokenizer = AutoTokenizer.from_pretrained(FLAN_T5_ID)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_T5_ID).to(DEVICE)
t5_model.eval()

print("✅ All models loaded successfully into VRAM!")

1. Loading Stage 1: Qwen-VL + LoRA...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

2. Loading Stage 2: MiniLM Semantic Retriever...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

DebertaV2Model LOAD REPORT from: cross-encoder/nli-deberta-v3-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
classifier.bias                 | UNEXPECTED |  | 
deberta.embeddings.position_ids | UNEXPECTED |  | 
pooler.dense.weight             | UNEXPECTED |  | 
pooler.dense.bias               | UNEXPECTED |  | 
classifier.weight               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4. Loading Stage 2: Flan-T5 Rationale Generator...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ All models loaded successfully into VRAM!


## Pruning & Retrieval Logic

In [4]:
import math, re, json, torch
import numpy as np

# ══════════════════════════════════════════════════════════════════════════════
# STAGE 1b — Post-processing helpers
# ══════════════════════════════════════════════════════════════════════════════

_BUCKET_RE = re.compile(r"^\d+[\+\-]?(\d+)?$")

def _is_bucket_name(name):
    return bool(_BUCKET_RE.match(str(name).strip()))

def _is_value_name(name):
    return bool(re.match(r"^[\d\.\s]+(kg|%|k|m|b|billion)?$",
                         str(name).strip().lower()))

def flatten_bucket_series(spec):
    if not isinstance(spec, dict): return spec
    series_list = spec.get("ser", [])
    buckets = [s for s in series_list
               if isinstance(s, dict) and _is_bucket_name(s.get("name", ""))]
    if len(buckets) < 2: return spec
    flat_data = []
    for bs in buckets:
        for pt in bs.get("data", []):
            if isinstance(pt, (list, tuple)) and len(pt) >= 2:
                flat_data.append([str(pt[0]), pt[1]])
    spec["ser"] = [{"name": spec.get("title", "entities"), "data": flat_data}] +                   [s for s in series_list if s not in buckets]
    spec["_bucket_merged"] = True
    return spec

def fix_series_naming(spec):
    if not isinstance(spec, dict): return spec
    row_labels = spec.get("topo", {}).get("row_labels", [])                  if isinstance(spec.get("topo"), dict) else []
    for pos, ser in enumerate(spec.get("ser", [])):
        if not isinstance(ser, dict): continue
        name = str(ser.get("name", ""))
        if not _is_value_name(name): continue
        data = ser.get("data", [])
        if not data: continue
        new_name = (row_labels[pos] if row_labels and pos < len(row_labels)
                    else str(data[0][0]) if isinstance(data[0], (list, tuple)) else name)
        ser["name"] = new_name
        ser["_original_value_name"] = name
    return spec


# ── Dual-axis detection ───────────────────────────────────────────────────────

def _is_dual_axis(spec):
    """True when series use incompatible scales (e.g. $T vs raw headcount)."""
    series_list = spec.get("ser", [])
    if len(series_list) < 2: return False
    y_refs = {str(s.get("y_ref", "")) for s in series_list if isinstance(s, dict)}
    if len(y_refs) > 1 and "" not in y_refs: return True
    maxes = []
    for ser in series_list:
        if not isinstance(ser, dict): continue
        vals = []
        for pt in ser.get("data", []):
            if isinstance(pt, (list, tuple)) and len(pt) >= 2:
                try: vals.append(abs(float(pt[1])))
                except: pass
        if vals: maxes.append(max(vals))
    if len(maxes) >= 2:
        ratio = max(maxes) / min(maxes) if min(maxes) > 0 else float("inf")
        if ratio > 20: return True
    return False


# ── Within-series temporal rel (for dual-axis charts) ────────────────────────

def _within_series_temporal_rel(series_list):
    """Emit first→last direction, peak/trough, and turning points per series."""
    synthesised = []
    for ser in series_list:
        if not isinstance(ser, dict): continue
        sname = ser.get("name", "?")
        pts = []
        for pt in ser.get("data", []):
            if isinstance(pt, (list, tuple)) and len(pt) >= 2:
                try: pts.append((str(pt[0]), float(pt[1])))
                except: pass
        if len(pts) < 2: continue
        first_lbl, first_val = pts[0]
        last_lbl,  last_val  = pts[-1]
        direction = ("increased" if last_val > first_val
                     else "decreased" if last_val < first_val else "stayed flat")
        delta = round(abs(last_val - first_val), 4)
        synthesised.append({
            "ref": f"{sname}: {first_lbl} to {last_lbl}",
            "relation": (f"For '{sname}', the value {direction} from "
                         f"{first_val} ({first_lbl}) to {last_val} ({last_lbl}), "
                         f"a change of {delta}."),
            "ranking": ([last_lbl, first_lbl] if last_val > first_val
                        else [first_lbl, last_lbl]),
        })
        max_pt = max(pts, key=lambda x: x[1])
        min_pt = min(pts, key=lambda x: x[1])
        if max_pt[0] != min_pt[0]:
            synthesised.append({
                "ref": f"{sname}: peak vs trough",
                "relation": (f"For '{sname}', the highest value is "
                             f"{max_pt[1]} ({max_pt[0]}) and the lowest is "
                             f"{min_pt[1]} ({min_pt[0]})."),
                "ranking": [max_pt[0], min_pt[0]],
            })
        for k in range(1, len(pts) - 1):
            pv, cv, nv = pts[k-1][1], pts[k][1], pts[k+1][1]
            if (cv > pv and cv > nv) or (cv < pv and cv < nv):
                turn = "peak" if cv > pv else "trough"
                synthesised.append({
                    "ref": f"{sname}: {turn} at {pts[k][0]}",
                    "relation": (f"For '{sname}', there is a {turn} at "
                                 f"{pts[k][0]} ({cv}), between "
                                 f"{pts[k-1][0]} ({pv}) and {pts[k+1][0]} ({nv})."),
                    "ranking": [],
                })
    return synthesised


# ── Single-series ranked rel (for horizontal bar / leaderboard charts) ────────

def _single_series_ranked_rel(series_list):
    """
    Top-vs-bottom, consecutive top-5, AND explicit rank+value statements.
    The rank statements directly falsify compound rank+threshold claims
    like "Process automation is the most common, used by over 70%".
    """
    synthesised = []
    ser = series_list[0]
    if not isinstance(ser, dict): return synthesised
    flat = {}
    for pt in ser.get("data", []):
        if isinstance(pt, (list, tuple)) and len(pt) >= 2:
            try: flat[str(pt[0])] = float(pt[1])
            except: pass
    if not flat: return synthesised
    ranked = sorted(flat.items(), key=lambda x: x[1], reverse=True)
    seen   = set()

    def _add(h_name, h_val, l_name, l_val):
        key = (h_name, l_name)
        if key in seen: return
        seen.add(key)
        delta = round(h_val - l_val, 4)
        synthesised.append({
            "ref": f"{h_name} vs {l_name}",
            "relation": (f"'{h_name}' ({h_val}) is higher than "
                         f"'{l_name}' ({l_val}) by {delta}."),
            "ranking": [h_name, l_name],
        })

    if len(ranked) >= 2:
        _add(ranked[0][0], ranked[0][1], ranked[-1][0], ranked[-1][1])
    for k in range(min(4, len(ranked) - 1)):
        _add(ranked[k][0], ranked[k][1], ranked[k+1][0], ranked[k+1][1])

    top_name = ranked[0][0]
    for rank_idx, (name, val) in enumerate(ranked, start=1):
        synthesised.append({
            "ref": f"rank:{name}",
            "relation": (f"'{name}' is ranked #{rank_idx} with a value of {val}. "
                         f"The top-ranked item is '{top_name}' ({ranked[0][1]})."),
            "ranking": [name],
        })
    return synthesised


# ── Master build_rel_if_missing ───────────────────────────────────────────────

def build_rel_if_missing(spec):
    """
    Three cases:
      A. Dual-axis / multi-scale  → within-series temporal comparisons
      B. Normal multi-series      → cross-series at shared x-labels
      C. Single flat series       → ranked rel with explicit rank statements
    """
    if not isinstance(spec, dict): return spec
    if spec.get("rel"):            return spec
    series_list = spec.get("ser", [])
    if not series_list:            return spec

    synthesised = []

    if len(series_list) > 1 and _is_dual_axis(spec):
        synthesised = _within_series_temporal_rel(series_list)

    elif len(series_list) > 1:
        groups = {}
        for ser in series_list:
            if not isinstance(ser, dict): continue
            sname = ser.get("name", "?")
            for pt in ser.get("data", []):
                if isinstance(pt, (list, tuple)) and len(pt) >= 2:
                    try:
                        groups.setdefault(str(pt[0]), []).append((sname, float(pt[1])))
                    except: pass
        for x_lbl, entries in groups.items():
            if len(entries) < 2: continue
            entries_sorted = sorted(entries, key=lambda e: e[1], reverse=True)
            top_name, top_val = entries_sorted[0]
            low_name, low_val = entries_sorted[-1]
            delta = round(top_val - low_val, 4)
            synthesised.append({
                "ref": x_lbl,
                "relation": (f"For '{x_lbl}', '{top_name}' ({top_val}) is higher than "
                             f"'{low_name}' ({low_val}) by {delta}."),
                "ranking": [e[0] for e in entries_sorted],
            })

    elif len(series_list) == 1:
        synthesised = _single_series_ranked_rel(series_list)

    if synthesised:
        spec["rel"] = synthesised
    return spec


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2a — Smart Downsampling
# ══════════════════════════════════════════════════════════════════════════════

def smart_downsample(raw_data, claim_text, max_points=15):
    if len(raw_data) <= max_points: return raw_data
    claim_lower = claim_text.lower()
    keep = {0, len(raw_data) - 1}

    def get_y(pt):
        if isinstance(pt, (list, tuple)):
            return float(pt[1]) if len(pt) >= 2 else float(pt[0])
        return float(pt)

    for i in range(1, len(raw_data) - 1):
        item = raw_data[i]
        if isinstance(item, (list, tuple)) and len(item) >= 1:
            if str(item[0]).lower() in claim_lower:
                keep.add(i); continue
        try:
            y_c, y_p, y_n = get_y(item), get_y(raw_data[i-1]), get_y(raw_data[i+1])
            if (y_c > y_p and y_c > y_n) or (y_c < y_p and y_c < y_n):
                keep.add(i)
        except: pass

    remaining = max_points - len(keep)
    if remaining > 0:
        avail = [i for i in range(len(raw_data)) if i not in keep]
        step  = max(1, len(avail) // remaining)
        keep.update(avail[::step][:remaining])
    return [raw_data[i] for i in sorted(keep)[:max_points]]


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2a — Semantic Retrieval
# ══════════════════════════════════════════════════════════════════════════════

def retrieve_top_k_series(claim, spec, k=3):
    series_list = spec.get("ser", [])
    if not isinstance(series_list, list) or len(series_list) <= k: return spec
    docs = [claim] + [json.dumps(s) for s in series_list]
    with torch.no_grad():
        embs = semantic_retriever.encode(docs, convert_to_tensor=True)
        sims = torch.nn.functional.cosine_similarity(embs[0:1], embs[1:])
    top_k = sims.topk(k).indices.sort().values.tolist()
    spec["ser"] = [series_list[i] for i in top_k]
    spec["is_truncated"] = True
    return spec


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2a — Pruning pipeline
# ══════════════════════════════════════════════════════════════════════════════

def prune_spec_for_encoder(raw_spec_dict, claim_text):
    """
    Full pipeline:
      1. Flatten bucket series
      2. Fix series naming
      3. Build rel if missing (with dual-axis + rank-statement awareness)
      4. Kill VLM hallucination loops
      5. Semantic top-k retrieval
      6. Downsample data points
    """
    if not isinstance(raw_spec_dict, dict): return {}
    cleaned = raw_spec_dict.copy()
    if not cleaned.get("ser"):              return {}

    cleaned = flatten_bucket_series(cleaned)
    cleaned = fix_series_naming(cleaned)
    cleaned = build_rel_if_missing(cleaned)

    def compress_val(v):
        if isinstance(v, float):
            return int(v) if v.is_integer() else round(v, 2)
        return v

    def sweep_for_loops(obj):
        if isinstance(obj, dict):
            for k, v in list(obj.items()):
                if k == "data": continue
                if isinstance(v, list) and all(isinstance(x, (str, int, float, bool)) for x in v):
                    obj[k] = [compress_val(x) for i, x in enumerate(v)
                               if i == 0 or x != v[i-1]][:20]
                elif isinstance(v, (dict, list)):
                    sweep_for_loops(v)

    cleaned.pop("legend", None)
    sweep_for_loops(cleaned)
    cleaned = retrieve_top_k_series(claim_text, cleaned, k=3)

    topo_type = (cleaned.get("topo", {}).get("type", "").lower()
                 if isinstance(cleaned.get("topo"), dict) else "")
    for ser in cleaned.get("ser", []):
        if not isinstance(ser, dict): continue
        for drop_key in ("ds", "is_subsampled", "critical_points_retained", "y_ref"):
            ser.pop(drop_key, None)
        if "pie" in topo_type:
            ser.pop("trend", None); ser.pop("stats", None)
        if "data" in ser and isinstance(ser["data"], list):
            rd = smart_downsample(ser["data"], claim_text, max_points=15)
            ser["data"] = [
                [compress_val(v) for v in pt] if isinstance(pt, list) else compress_val(pt)
                for pt in rd
            ]
    return cleaned


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2b — Linearisation
# ══════════════════════════════════════════════════════════════════════════════

def spec_to_prose(spec):
    """
    Convert pruned ChartSpec dict → natural-language premise.
    Kept identical to the training-notebook version so DeBERTa sees the
    same format at inference as during fine-tuning.
    """
    if not isinstance(spec, dict): return ""

    topo       = spec.get("topo", {})
    ctype      = topo.get("type",       "unknown") if isinstance(topo, dict) else "unknown"
    csub       = topo.get("sub",        "")        if isinstance(topo, dict) else ""
    row_labels = topo.get("row_labels", [])        if isinstance(topo, dict) else []

    type_str = f"{csub} {ctype}".strip() if csub else ctype
    parts = [f"Chart type: {type_str}"]

    if row_labels:
        parts.append(f"Categories: {', '.join(str(r) for r in row_labels)}")

    for ax in spec.get("axes", []):
        if isinstance(ax, dict):
            dom = ax.get("dom", [])
            if isinstance(dom, list) and len(dom) == 2:
                parts.append(f"Axis '{ax.get('name', '?')}' range: {dom[0]} to {dom[1]}")

    for ser in spec.get("ser", []):
        if isinstance(ser, str):
            try:    ser = json.loads(ser)
            except: continue
        if not isinstance(ser, dict): continue
        name  = ser.get("name",  "unnamed")
        data  = ser.get("data",  [])
        stats = ser.get("stats", {})
        trend = ser.get("trend", "")

        if isinstance(data, list):
            pts = ", ".join(
                f"({p[0]}: {p[1]})" if isinstance(p, (list, tuple)) and len(p) >= 2
                else str(p) for p in data[:15]
            )
        else:
            pts = str(data)

        line = f"Series '{name}': {pts}"
        if isinstance(stats, dict) and stats:
            mn = stats.get("min_value", stats.get("min", ""))
            mx = stats.get("max_value", stats.get("max", ""))
            if mn != "" and mx != "":
                line += f". Min={mn}, Max={mx}"
        if trend:
            trend_str = (trend.get("global_trend", str(trend))
                         if isinstance(trend, dict) else str(trend))
            if trend_str:
                line += f". Trend: {trend_str}"
        parts.append(line)

    for rel in spec.get("rel", []):
        if not isinstance(rel, dict): continue
        relation = rel.get("relation", "")
        ranking  = rel.get("ranking",  [])
        ref      = rel.get("ref",      "")
        if relation: parts.append(f"Comparison: {relation}")
        if ranking:  parts.append(
            f"Ranking at '{ref}': {' > '.join(str(r) for r in ranking)}"
        )

    return ". ".join(parts) + "."


print("✅ All Stage 1b / 2a / 2b helpers loaded.")


✅ All Stage 1b / 2a / 2b helpers loaded.


In [17]:
# (helpers moved to cell above)


In [18]:
# (retrieve_top_k_series moved to helpers cell above)


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — Spec Extraction
# ══════════════════════════════════════════════════════════════════════════════

def extract_chart_spec(image_path):
    """
    Extract Extended ChartSpec JSON via the fine-tuned QwenVL.
    Prompt is minimal — matching what the LoRA was trained on.
    """
    schema_prompt = "Extract the Extended ChartSpec JSON from this chart image."
    messages = [
        {"role": "system",
         "content": "You output strictly valid JSON with no markdown formatting."},
        {"role": "user", "content": [
            {"type": "image", "image": image_path, "max_pixels": 768 * 768},
            {"type": "text",  "text": schema_prompt},
        ]},
    ]
    text = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **inputs, max_new_tokens=2048,
            do_sample=False, repetition_penalty=1.1,
        )
    trimmed  = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    raw_text = qwen_processor.batch_decode(trimmed, skip_special_tokens=True)[0]
    clean    = re.sub(r'\}\]\}\],\"rel\"', '}],\"rel\"', raw_text)
    try:
        return json_repair.repair_json(clean, return_objects=True)
    except Exception:
        return {}


print("✅ extract_chart_spec loaded.")


✅ extract_chart_spec loaded.


## NLI Reasoning & Execution Loop

In [6]:
# --- The 5-Sample Continuous Simulation ---
simulation_samples = [
    {
        "image_path": "17979.jpeg",
        "title_description": "2020s: The Billionaires' Decade? Cumulative wealth and number of billionaires worldwide",
        "chart_type": "bar_chart",
        "claim": "The cumulative wealth of billionaires steadily decreased between 2020 and 2026.",
        "truth" : " (REFUTED claim)",
        "source": "Forbes World's Billionaires List | Statista",
        "source_url": "https://www.statista.com/chart/daily/"
    },
    {
        "image_path": "17979.jpeg",
        "title_description": "2020s: The Billionaires' Decade? Cumulative wealth and number of billionaires worldwide",
        "chart_type": "bar_chart",
        "claim": "The number of billionaires worldwide more than doubled from 2013 to 2026.",
        "truth" : " (SUPPORTED claim)",
        "source": "Forbes World's Billionaires List | Statista",
        "source_url": "https://www.statista.com/chart/daily/"
    },
    {
        "image_path": "35976.jpeg",
        "title_description": "Content Marketers Embrace AI in Content Creation: Share of respondents whose department applies AI tools for the following use cases",
        "chart_type": "horizontal_bar_chart",
        "claim": "Process automation is the most common use case for AI among content marketers, utilized by over 70% of respondents.",
        "truth": " (REFUTED claim)",
        "source": "Statista+ | Content Marketing Trend Study 2026",
        "source_url": "https://www.statista.com/chart/daily/"
    },
    {
        "image_path": "35976.jpeg",
        "title_description": "Content Marketers Embrace AI in Content Creation: Share of respondents whose department applies AI tools for the following use cases",
        "chart_type": "horizontal_bar_chart",
        "claim": "Over half of the surveyed content marketing professionals use AI tools for content creation, making it the top use case.",
        "truth" : " (SUPPORTED claim)",
        "source": "Statista+ | Content Marketing Trend Study 2026",
        "source_url": "https://www.statista.com/chart/daily/"
    },
    {
        "image_path": "35957.jpeg",
        "title_description": "Web Censorship Cases Rebound in 2025: Documented cases of government-imposed online censorship worldwide",
        "chart_type": "clustered_bar_chart",
        "claim": "The number of new web censorship cases steadily decreased every year from 2022 to 2025.",
        "truth": " (REFUTED claim)",
        "source": "Surfshark | Statista",
        "source_url": "https://www.statista.com/chart/daily/"
    },
    {
        "image_path": "35957.jpeg",
        "title_description": "Web Censorship Cases Rebound in 2025: Documented cases of government-imposed online censorship worldwide",
        "chart_type": "clustered_bar_chart",
        "claim": "There were more new web censorship cases reported in 2025 than in any other year shown on the chart.",
        "truth" : " (SUPPORTED claim)",
        "source": "Surfshark | Statista",
        "source_url": "https://www.statista.com/chart/daily/"
    }
]

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2c — Evidence Extraction
# ══════════════════════════════════════════════════════════════════════════════

def extract_attended_sentence(premise_str, input_ids_1d, attn_row, tokenizer):
    """
    Return the highest-attended DATA sentence from the premise.
    Skips description/header lines; uses float-safe ". " splitting.
    """
    tokens  = tokenizer.convert_ids_to_tokens(input_ids_1d.tolist())
    attn_np = attn_row.float().cpu().numpy()

    sep_indices = [i for i, t in enumerate(tokens) if t in ("[SEP]", "</s>")]
    if len(sep_indices) < 2: return "No specific data found."

    premise_start  = sep_indices[0] + 1
    premise_end    = sep_indices[1]
    premise_tokens = tokens[premise_start:premise_end]
    premise_attn   = attn_np[premise_start:premise_end]
    if not premise_tokens: return "No specific data found."

    premise_decoded = [
        tokenizer.convert_tokens_to_string([t]).strip() for t in premise_tokens
    ]

    raw_sentences = re.split(r'\. ', premise_str)
    sentences = [s.strip() for s in raw_sentences if s.strip()]
    if not sentences: return "No specific data found."

    DATA_PREFIXES = ("series", "comparison", "ranking", "axis",
                     "trend", "min=", "max=", "categories")

    data_sentences = [s for s in sentences
                      if any(s.lower().startswith(p) for p in DATA_PREFIXES)]
    candidates = data_sentences if data_sentences else sentences

    best_sentence, best_score = candidates[0], -1.0
    for sentence in candidates:
        sent_lower = sentence.lower()
        token_scores = [
            attn for tok_str, attn in zip(premise_decoded, premise_attn)
            if tok_str and len(tok_str) > 1 and tok_str.lower() in sent_lower
        ]
        if token_scores:
            score = float(np.mean(token_scores))
            if score > best_score:
                best_score, best_sentence = score, sentence

    return best_sentence.strip() + "."


# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2d — Rationale Generation
# ══════════════════════════════════════════════════════════════════════════════

def generate_rationale(claim, pred_label, evidence):
    """Generate a grounded rationale with Flan-T5."""
    verdict = "supported" if pred_label == 1 else "refuted"
    prompt  = (
        f"You are a chart fact-checking assistant. "
        f"The following claim has been verified as {verdict}.\n"
        f"Claim: {claim}\n"
        f"Chart evidence: {evidence}\n"
        f"Write one concise sentence explaining what the specific chart numbers show "
        f"and why the claim is therefore {verdict}. "
        f"Cite the actual values from the evidence. "
        f"Do NOT restate or paraphrase the claim.\n"
        f"Explanation:"
    )
    inputs = t5_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = t5_model.generate(
            **inputs, max_new_tokens=80, min_new_tokens=15,
            do_sample=False, num_beams=4,
            no_repeat_ngram_size=3, repetition_penalty=1.3,
        )
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


# ══════════════════════════════════════════════════════════════════════════════
# MAIN SIMULATION LOOP
# ══════════════════════════════════════════════════════════════════════════════

output_json_path   = "simulation_results.json"
simulation_results = []

print("🚀 Starting Continuous Chart Verification Pipeline...\n" + "="*60)

for idx, sample in enumerate(simulation_samples):
    print(f"\n--- Processing Sample {idx + 1} ---")
    if not os.path.exists(sample["image_path"]):
        print(f"⚠️  Image not found: {sample['image_path']}. Skipping.")
        continue

    # 1. Extract raw spec from image
    raw_spec = extract_chart_spec(sample["image_path"])

    # 1b. Post-process: flatten buckets → fix naming → build rel
    raw_spec = flatten_bucket_series(raw_spec)
    raw_spec = fix_series_naming(raw_spec)
    raw_spec = build_rel_if_missing(raw_spec)

    # 2. Prune & Retrieve
    full_context = f"{sample['claim']} {sample['title_description']}"
    pruned_spec  = prune_spec_for_encoder(raw_spec, full_context)

    # 3. Linearise
    prose_spec  = spec_to_prose(pruned_spec)
    premise_str = (
        f"Chart Description: {sample['title_description']}. "
        f"Chart Type: {sample['chart_type']}. "
        f"Data: {prose_spec}"
    )

    # 4. Verify Claim
    enc  = nli_tokenizer(
        text=sample["claim"], text_pair=premise_str,
        max_length=512, truncation="only_second", return_tensors="pt"
    )
    ids, mask = enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits, mean_attn = nli_model(ids, mask, return_attentions=True)
        pred_val = int((torch.sigmoid(logits) >= 0.5).long().item())

    cls_attn = mean_attn[0, 0, :]

    # 5. Extract Evidence & Generate Rationale
    evidence  = extract_attended_sentence(premise_str, ids[0], cls_attn, nli_tokenizer)
    rationale = generate_rationale(sample["claim"], pred_val, evidence)

    prediction_label = "SUPPORTED" if pred_val == 1 else "REFUTED"

    # 6. Console output
    print(f"Prosed Spec          : {prose_spec}")
    print(f"Claim                : {sample['claim']}")
    print(f"Truth                : {sample['truth']}")
    print(f"DeBERTa Prediction   : {prediction_label}")
    print(f"Extracted Evidence   : {evidence}")
    print(f"Flan-T5 Rationale    : {rationale}")

    # 7. Incremental save
    simulation_results.append({
        "sample_id":         idx + 1,
        "image_path":        sample["image_path"],
        "title_description": sample["title_description"],
        "chart_type":        sample["chart_type"],
        "claim":             sample["claim"],
        "truth":             sample["truth"],
        "prosed_spec":       prose_spec,
        "prediction":        prediction_label,
        "evidence":          evidence,
        "rationale":         rationale,
        "pruned_spec":       pruned_spec,
        "raw_spec":          raw_spec,
    })

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(simulation_results, f, indent=4, ensure_ascii=False)
    print(f"💾 Saved Sample {idx + 1} to {output_json_path}")

print("\n" + "="*60 + "\n✅ Simulation Complete!")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🚀 Starting Continuous Chart Verification Pipeline...

--- Processing Sample 1 ---
Prosed Spec          : Chart type: vertical bar. Axis 'y_axis' range: 4.15 to 20.85. Axis 'y_axis_1' range: 1226 to 3428. Series 'Billionaire Wealth (in $T)': (2013: 5.4), (2014: 6.4), (2015: 7.1), (2016: 6.5), (2017: 7.7), (2018: 9.1), (2019: 8.7), (2020: 8), (2021: 13.1), (2022: 12.7), (2023: 12.2), (2024: 14.2), (2025: 16.1), (2026: 20.1). Min=5.4, Max=20.1. Trend: fluctuating. Series 'Number of billionaires': (2013: 1426), (2014: 1645), (2015: 1826), (2016: 1810), (2017: 2043), (2018: 2208), (2019: 2153), (2020: 2095), (2021: 2755), (2022: 2668), (2023: 2640), (2024: 2781), (2025: 3028), (2026: 3428). Min=1426, Max=3428. Trend: fluctuating. Comparison: At ref point 2013, 'Billionaire Wealth (in $T)' has a higher value (5.4) than 'Number of billionaires' (1426) by 1420.6.. Ranking at '2013': Number of billionaires > Billionaire Wealth (in $T). Comparison: At ref point 2020, 'Billionaire Wealth (in $T)'